# Black-Litterman

Two ingredients, blended Bayesian-style:

- **The prior:** the market's implied equilibrium returns $\Pi$. If you hold no views, you should hold the market portfolio — and BL guarantees exactly that.
- **The views:** your specific opinions ("asset A will outperform B by 2%"), each with a confidence level. These nudge the posterior away from equilibrium *only* in the directions you have an opinion, and *only* as much as your confidence warrants.

In [1]:
from datetime import date, timedelta
import numpy as np
import yfinance as yf
from util_yahoo_finance import get_returns_and_covariance, get_market_caps

## Reverse Optimization for the Prior

Given expected returns $\mu$, Markowitz finds the optimal weights. The mean-variance utility solution was:

$$
w = \frac{1}{\gamma}\Sigma^{-1}(\mu - \lambda\mathbf{1})
$$

Expected returns in, optimal weights out.

**Assume** the market portfolio $w_{mkt}$ (market-cap weights) is already mean-variance optimal — which it must be in CAPM equilibrium, since everyone holds the same tangency portfolio. Then ask: **what expected returns would make these observed weights optimal?**

Solve the first-order condition for $\mu$ instead of $w$. From the unconstrained mean-variance optimum, the optimal weights satisfy:

$$
w_{mkt} \propto \Sigma^{-1}\Pi
$$

Invert to solve for the implied returns:

$$
\boxed{\Pi = \delta\,\Sigma\, w_{mkt}}
$$

This is **reverse optimization**. $\Pi$ is the vector of **equilibrium implied returns** — the returns the market collectively believes in, revealed by how it's allocated. $\delta$ is the market's risk-aversion coefficient (a scalar, typically calibrated as $\delta = \frac{\text{market excess return}}{\text{market variance}}$, often around 2.5).

So the prior distribution is $\mu \sim N(\Pi, \tau\Sigma)$ — centered at the equilibrium returns $\Pi$, with uncertainty $\tau\Sigma$ ($\tau$ a small scalar, since the equilibrium is a fairly confident anchor).

In [2]:
def market_cap_weights(tickers: list[str]) -> np.ndarray:
    caps = get_market_caps(tickers)
    values = np.array(list(caps.values()), dtype=float)
    return values / values.sum()

From the CAPM derivation, $\delta$ is the market's price of risk:
$$\delta = \frac{\mu_m - r_f}{\sigma_m^2}​​$$
Market excess return over market variance. Plug in long-run US equity numbers — say a 5-6% equity risk premium and ~15-16% annual volatility, so $\sigma_m^2 \approx 0.024$ — and you get $\delta \approx 0.055/0.024 \approx 2.3$ to $2.5$. That's where the number comes from. It's not a universal constant; it's an empirical estimate of how much extra return the market historically demanded per unit of variance.

In [3]:
def estimate_delta(proxy: str = "QQQ", years: int = 3, rf: float = 0.05) -> float:
    """
    Estimate the market risk-aversion coefficient δ = (μ_m - rf) / σ²_m
    from historical data of a market proxy (default: QQQ).

    Parameters
    ----------
    proxy : ticker for the market portfolio proxy
    years : lookback window in years
    rf    : annualized risk-free rate

    Returns
    -------
    delta : scalar risk-aversion coefficient
    """
    start = (date.today() - timedelta(days=365 * years)).isoformat()
    prices = yf.download(proxy, start=start, auto_adjust=True, progress=False)["Close"].squeeze()
    log_rets = np.log(prices / prices.shift(1)).dropna()

    mu_m     = float(log_rets.mean()) * 252        # annualized mean
    sigma2_m = float(log_rets.var())  * 252        # annualized variance

    delta = (mu_m - rf) / sigma2_m
    print(f"{proxy} ({years}y)  μ={mu_m:.2%}  σ={np.sqrt(sigma2_m):.2%} sr={mu_m / np.sqrt(sigma2_m):.2f} δ={delta:.4f}")
    return delta

delta = estimate_delta(proxy="QQQ", years=3, rf=0.05)

QQQ (3y)  μ=25.06%  σ=19.58% sr=1.28 δ=5.2334


Attilio Meucci and others argue the whole $\tau$ business is a confusion, and you should just set $\tau = 1$ and fold all the uncertainty scaling into $\Omega$ instead. Their point: having two separate uncertainty knobs ($\tau$ for the prior, $\Omega$ for the views) is redundant, since only the *ratio* of prior precision to view precision matters. Set $\tau = 1$ and express everything through $\Omega$.

In [6]:
def market_implied_returns(w_mkt: np.ndarray, cov: np.ndarray,
                           rf: float = 0.05, delta: float = None) -> np.ndarray:
    """
    Compute Black-Litterman equilibrium (market-implied) excess returns.

        Π = δ Σ w_mkt

    Parameters
    ----------
    w_mkt : (n,) market-cap weights, sum to 1
    cov   : (n, n) annualized covariance matrix
    rf    : annualized risk-free rate (used only when delta is inferred)
    delta : risk-aversion coefficient; if None, inferred from the market
            portfolio's Sharpe ratio assuming μ_m ≈ rf + δ σ²_m

    Returns
    -------
    pi : (n,) vector of implied annualized excess returns
    """
    w_mkt = np.asarray(w_mkt, dtype=float)
    cov   = np.asarray(cov,   dtype=float)

    if delta is None:
        # Back out δ = (μ_m - rf) / σ²_m from a target market Sharpe of 0.4
        # (a widely used empirical estimate for developed equity markets)
        sigma2_m = float(w_mkt @ cov @ w_mkt)
        target_sharpe = 0.4
        delta = target_sharpe / np.sqrt(sigma2_m)
        print(f"Delta: {delta:.2f}")

    pi = delta * cov @ w_mkt
    return pi

tickers = ["GOOGL", "NVDA", "AAPL"]
returns, cov      = get_returns_and_covariance(tickers, start="2020-01-01")
w_mkt             = market_cap_weights(tickers)
pi                = market_implied_returns(w_mkt, cov, rf=0.05, delta=3)

for t, r in zip(tickers, pi):
    print(f"{t}: {r:.2%}")

GOOGL: 26.13%
NVDA: 47.63%
AAPL: 25.39%


## Views as the Likelihood

**The picking matrix $P$** ( $k \times n$, one row per view). Each row selects which assets a view is about. Two view types:

- **Absolute view:** "Asset 3 will return 5%." The row is all zeros except a 1 in position 3: $[0\ 0\ 1\ 0\ \dots]$.
- **Relative view:** "Asset 1 outperforms asset 2 by 2%." The row has $+1$ and $-1$: $[1\ -1\ 0\ \dots]$. Relative views sum to zero across the row — they're long-short, expressing a spread, not a level.

**The view vector $Q$** ($k \times 1$). The magnitude of each view — the 5%, the 2%. One entry per row of $P$.

**The uncertainty matrix $\Omega$** ( $k \times k$, usually diagonal). How confident you are in each view. Small $\Omega_{ii}$ = tight confidence (the view strongly pulls the posterior); large $\Omega_{ii}$ = vague (the view barely moves anything). This is the precision knob from the conjugate-updating session — $\Omega^{-1}$ is the view precision.

**Assembling the likelihood.** The views say: the true returns $\mu$, viewed through $P$, should equal $Q$ — up to noise:

$$
P\mu = Q + \epsilon, \qquad \epsilon \sim N(0, \Omega)
$$

Read that as a statement about $\mu$: "the linear combination $P\mu$ is observed to be $Q$, with uncertainty $\Omega$." That is *exactly* the multivariate likelihood — $y \mid \theta \sim N(H\theta, R)$ — with $\theta = \mu$, $H = P$, $y = Q$, $R = \Omega$. The picking matrix is the observation matrix; your views are the noisy observations of the parameter.

**The one subtlety worth flagging — calibrating $\Omega$.** Pulling confidence numbers out of thin air is the practical weak point of BL. The standard convention (He-Litterman) ties view uncertainty to the prior's own scale:

$$
\Omega = \text{diag}(P(\tau\Sigma)P^T)
$$

This says "each view's uncertainty matches the equilibrium uncertainty along that view's direction" — a neutral default where the view and the prior get balanced weight. You scale it up or down per view to express stronger or weaker conviction. The point: $\Omega$ should be calibrated relative to $\tau\Sigma$, not picked arbitrarily, since the two precisions are what get weighed against each other.

In [7]:
P = np.array([[1,0,0],[0,1,0],[0,0,1]])
Q = np.array([0.25, 0.5, 0.2])
c = np.array([0.9, 0.8, 0.75])

def omega_from_confidence(P, tau_sigma, confidences):
    """
    Build diagonal Omega from per-view confidence percentages in (0,1).
    Uses the proportional (odds) mapping relative to the He-Litterman base.
        Omega_ii = (P tau_sigma P^T)_ii * (1 - c) / c
    """
    base = np.diag(P @ tau_sigma @ P.T)        # neutral He-Litterman scale
    c = np.asarray(confidences, dtype=float)
    if np.any((c <= 0) | (c >= 1)):
        raise ValueError("confidences must be strictly between 0 and 1")
    return np.diag(base * (1 - c) / c)
omega = omega_from_confidence(P, cov, c)

## The Master Formula

**Posterior precision** (precisions add):

$$
\Sigma_{post}^{-1} = (\tau\Sigma)^{-1} + P^T\Omega^{-1}P
$$

**Posterior mean — the Black-Litterman master formula:**

$$
\boxed{\mu_{BL} = \left[(\tau\Sigma)^{-1} + P^T\Omega^{-1}P\right]^{-1}\left[(\tau\Sigma)^{-1}\Pi + P^T\Omega^{-1}Q\right]}
$$

Read it the way you read the scalar update: posterior covariance (the leading inverse = inverse of total precision) times the precision-weighted sum of prior and views.$(\tau\Sigma)^{-1}\Pi$ is "equilibrium weighted by its confidence," $P^T\Omega^{-1}Q$ is "views weighted by their confidence, routed back to asset space through $P^T$."

In [8]:
def black_litterman(sigma, w_mkt, P, Q, omega=None, tau=1, delta=2.5):
    pi = delta * (sigma @ w_mkt)            # equilibrium prior mean
    tau_sigma = tau * sigma                 # prior covariance on the mean
    if omega is None:
        omega = np.diag(np.diag(P @ tau_sigma @ P.T))   # He-Litterman default
    # precision-weighted posterior (the master formula)
    prior_prec = np.linalg.inv(tau_sigma)
    view_prec  = P.T @ np.linalg.inv(omega) @ P
    post_cov   = np.linalg.inv(prior_prec + view_prec)
    mu_bl = post_cov @ (prior_prec @ pi + P.T @ np.linalg.inv(omega) @ Q)
    return mu_bl
black_litterman(cov, w_mkt, P, Q, omega=omega, tau=1, delta=5)

array([0.25823294, 0.52457576, 0.23649534])

$\mu_{BL}$ is just a better expected-return vector. Drop it into mean-variance utility:

$$
\max_w \ \mu_{BL}^Tw - \frac{\gamma}{2}w^T\Sigma w \quad \text{s.t.} \quad \mathbf{1}^Tw = 1
$$

Some implementations also use the posterior covariance $\Sigma + \Sigma_{post}$ in place of $\Sigma$ (accounting for estimation uncertainty in returns), but the standard version uses $\mu_{BL}$ with the original $\Sigma$. The output weights are stable and sensible because $\mu_{BL}$ is anchored to equilibrium — no more error-maximization blowup.

**The full pipeline, end to end:**

1. Start with market-cap weights $w_{mkt}$ and covariance $\Sigma$
2. Reverse-optimize: $\Pi = \delta\Sigma w_{mkt}$ (the prior)
3. Specify views: $P$, $Q$, $\Omega$ (the likelihood)
4. Blend: $\mu_{BL}$ via the master formula (the posterior)
5. Optimize: feed $\mu_{BL}$ into Markowitz → final weights